# 01 - Data Preparation

Builds `data/processed/model_inputs.csv` from the two raw sources.

**Key caveat (repeat this in any write-up):** the two datasets share no key. Lead time is *sampled* per store-item pair from the logistics dataset's empirical Vehicle-Type distribution (weighted by real observation frequency, seed=42, reproducible) — not a real 1:1 join.

In [1]:
import sys
sys.path.append('..')
from src.data_prep import build_lead_time_stats, build_demand_stats, \
    sample_lead_time_per_pair, build_model_inputs, COST_ASSUMPTIONS
from src.logging_config import get_logger

logger = get_logger(__name__)
import pandas as pd

## Step 1 - Lead time from the logistics tracking dataset

`lead_time_days = Trip End Date - Booking Date`, filtered to remove:
- values <= 0 or < 1 hour (GPS ping artifacts, not real trips)
- top 1% outliers (a handful of abnormal trips that would skew std)

In [2]:
lead_time_stats = build_lead_time_stats()
lead_time_stats.sort_values('n_obs', ascending=False).head(15)

2026-08-28 21:13:18 | INFO     | src.data_prep | Lead time: kept 2853/3585 rows after noise filtering (upper cap = 47.0 days)


,group,lead_time_mean_days,lead_time_std_days,n_obs
0,OVERALL,6.399401,6.235456,2853
27,32 FT Multi-Axle 14MT - HCV,5.695536,4.077062,895
31,40 FT 3XL Trailer 35MT,7.473354,7.005714,637
30,32 FT Single-Axle 7MT - HCV,6.434025,5.331951,457
33,40 FT Flat Bed Multi-Axle 27MT - Trailer,7.233716,7.373434,213
28,32 FT Multi-Axle MXL 18MT,6.487616,3.693058,84
10,19 FT Open 7MT - MCV,17.961898,10.080517,77
15,20 FT SXL Container,4.933651,2.524618,42
32,40 FT Flat Bed Double-Axle 21MT - Trailer,6.950228,8.552541,41
20,24 FT SXL Container,4.247764,4.668449,35


## Step 2 - Demand distribution per (store, item)

913,000 rows, 500 pairs, 1,826 continuous daily observations each - dense enough to trust mean/std estimates directly (no smoothing needed).

In [3]:
demand_stats = build_demand_stats()
print(demand_stats['suggested_distribution'].value_counts())
demand_stats.describe()

suggested_distribution
Normal_approx    500
Name: count, dtype: int64


,store,item,demand_mean,demand_std,demand_min,demand_max,n_days,dispersion_index
count,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.0,500.000000
mean,5.500000,25.500000,52.250287,14.701935,15.968000,104.812000,1826.0,4.152787
std,2.875158,14.445322,24.062536,5.976603,9.840969,43.919318,0.0,1.452220
min,1.000000,1.000000,12.733844,4.731280,0.000000,31.000000,1826.0,1.757915
25%,3.000000,13.000000,30.964266,9.449078,8.000000,67.000000,1826.0,2.871932
50%,5.500000,25.500000,50.030942,14.142238,15.000000,101.000000,1826.0,4.020325
75%,8.000000,38.000000,69.348987,18.857686,23.000000,137.000000,1826.0,5.152435
max,10.000000,50.000000,112.638007,29.676051,44.000000,231.000000,1826.0,7.822677


## Step 3 - Sample lead time per pair (weighted by real Vehicle Type frequency)

In [4]:
sampled_lt = sample_lead_time_per_pair(lead_time_stats, n_pairs=len(demand_stats))
sampled_lt['lead_time_mean_days'].describe()

2026-08-28 21:13:19 | INFO     | src.data_prep | Lead time: 24 Vehicle Types eligible (>= 5 obs) as sampling source


count    500.000000
mean       6.749359
std        2.538857
min        0.606377
25%        5.695536
50%        6.434025
75%        7.473354
max       17.961898
Name: lead_time_mean_days, dtype: float64

## Step 4 - Cost assumptions (explicitly NOT derived from data — document as assumptions)

In [5]:
COST_ASSUMPTIONS

{'holding_cost_rate_annual': 0.2,
 'order_cost_fixed': 50.0,
 'target_service_level': 0.95,
 'unit_cost_placeholder': 10.0}

## Step 5 - Build and save the final model_inputs.csv

In [6]:
model_inputs = build_model_inputs(save=True)
model_inputs.head()

2026-08-28 21:13:19 | INFO     | src.data_prep | Starting model_inputs build


2026-08-28 21:13:21 | INFO     | src.data_prep | Lead time: kept 2853/3585 rows after noise filtering (upper cap = 47.0 days)


2026-08-28 21:13:22 | INFO     | src.data_prep | Lead time: 24 Vehicle Types eligible (>= 5 obs) as sampling source


2026-08-28 21:13:22 | INFO     | src.data_prep | Saved model_inputs.csv (500 rows) to C:\Users\Huy\Documents\inventory-optimization-project (1)\data\processed\model_inputs.csv


,store,item,demand_mean,demand_std,demand_min,demand_max,n_days,dispersion_index,suggested_distribution,assigned_vehicle_type,lead_time_mean_days,lead_time_std_days,holding_cost_rate_annual,order_cost_fixed,target_service_level,unit_cost_placeholder
0,1,1,19.971522,6.741022,4,50,1826,2.275309,Normal_approx,40 FT 3XL Trailer 35MT,7.473354,7.005714,0.2,50.0,0.95,10.0
1,1,2,53.148959,15.005779,13,115,1826,4.236648,Normal_approx,32 FT Multi-Axle 14MT - HCV,5.695536,4.077062,0.2,50.0,0.95,10.0
2,1,3,33.208105,10.072529,8,70,1826,3.055153,Normal_approx,40 FT 3XL Trailer 35MT,7.473354,7.005714,0.2,50.0,0.95,10.0
3,1,4,19.956188,6.640618,4,43,1826,2.209731,Normal_approx,40 FT 3XL Trailer 35MT,7.473354,7.005714,0.2,50.0,0.95,10.0
4,1,5,16.612815,5.672102,3,37,1826,1.936622,Normal_approx,22 FT Closed Container,1.097816,1.034386,0.2,50.0,0.95,10.0


In [7]:
# Quick visual sanity check
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
model_inputs['demand_mean'].hist(bins=30, ax=axes[0])
axes[0].set_title('Demand mean distribution across 500 pairs')
model_inputs['lead_time_mean_days'].hist(bins=30, ax=axes[1])
axes[1].set_title('Sampled lead time distribution across 500 pairs')
plt.tight_layout()
plt.show()

C:\Users\Huy\AppData\Local\Temp\ipykernel_17052\613080829.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
